# LangGraph Agentic RAG

This notebook runs the real document QA pipeline through all four Self-RAG reflections. It loads indexed chunks, retrieves and reranks candidates, filters irrelevant passages, verifies answer support, and checks answer utility.

The graph prepares bounded conversation context, decides whether retrieval is needed, rewrites the query, and then runs the existing RAG query path:

```text
START -> context_manager -> retrieval_gate[Ret] -> query_rewriter -> retrieve -> grade_relevance[Rel] -> generate -> verify_support[Sup] -> verify_utility[Use] -> persist/abstain -> END
```

This notebook covers [Ret], [Rel], [Sup], and [Use]. It does not implement regeneration, retrieval retry, or citation repair.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'src').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from within the project directory tree.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/humengqing/Documents/Code/VSCode/doc-qa-agent


In [2]:
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import Markdown, display

from src.agent.context import ContextManager
from src.agent.graph import build_agent_graph, invoke_agent_graph
from src.agent.relevance import LLMRelevanceGrader
from src.agent.rewrite import LLMQueryRewriter
from src.agent.routes import LLMRetrievalGate
from src.agent.support import LLMSupportVerifier
from src.agent.utility import LLMUtilityVerifier
from src.core.config import Config
from src.core.logger import setup_logging
from src.pipeline.query_runtime import build_query_pipeline

## Build the real RAG pipeline

This cell uses the same online query-runtime construction path as `main.py`. It loads existing chunks and document embeddings from Chroma, then rebuilds only the in-memory BM25 index. It does not parse documents or re-embed document chunks.

In [3]:
config = Config()
setup_logging(config)
pipeline = build_query_pipeline(config)

checkpointer = InMemorySaver()
graph = build_agent_graph(
    pipeline,
    retrieval_gate=LLMRetrievalGate(pipeline.llm),
    context_manager=ContextManager(config),
    query_rewriter=LLMQueryRewriter(pipeline.llm),
    relevance_grader=LLMRelevanceGrader(pipeline.llm),
    support_verifier=LLMSupportVerifier(pipeline.llm),
    utility_verifier=LLMUtilityVerifier(pipeline.llm),
    checkpointer=checkpointer,
)

2026-08-24 14:33:46,376 | INFO | src | Logging configured with level INFO


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-08-24 14:33:51,251 | INFO | src.retrieval.embedder | Configured local embedding model: BAAI/bge-large-en-v1.5
2026-08-24 14:33:51,301 | INFO | src.retrieval.vector_store | Loaded 306 indexed chunk(s) from collection doc_chunks
2026-08-24 14:33:51,311 | INFO | src.retrieval.bm25_retriever | Built BM25 index for 306 chunk(s)
2026-08-24 14:33:51,317 | INFO | src.retrieval.reranker | Configured scadsai reranker: Qwen/Qwen3-Reranker-4B


## Visualize the LangGraph workflow

`draw_mermaid()` exposes the compiled graph structure. LangSmith records executions of this graph when `LANGSMITH_TRACING=true` and `LANGSMITH_API_KEY` are configured in `.env`.

In [4]:
mermaid_graph = graph.get_graph().draw_mermaid()
display(Markdown(f'```mermaid\n{mermaid_graph}\n```'))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieval_gate(retrieval_gate)
	context_manager(context_manager)
	query_rewriter(query_rewriter)
	retrieve(retrieve)
	grade_relevance(grade_relevance)
	generate(generate)
	verify_support(verify_support)
	verify_utility(verify_utility)
	abstain(abstain)
	persist_turn(persist_turn)
	__end__([<p>__end__</p>]):::last
	__start__ --> context_manager;
	abstain --> persist_turn;
	context_manager --> retrieval_gate;
	generate --> verify_support;
	grade_relevance -.-> abstain;
	grade_relevance -.-> generate;
	query_rewriter --> retrieve;
	retrieval_gate -.-> abstain;
	retrieval_gate -. &nbsp;retrieve&nbsp; .-> query_rewriter;
	retrieve --> grade_relevance;
	verify_support -.-> abstain;
	verify_support -. &nbsp;verify&nbsp; .-> verify_utility;
	verify_utility -.-> abstain;
	verify_utility -. &nbsp;persist&nbsp; .-> persist_turn;
	persist_turn --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## Run a two-turn conversation

Both calls use the same `thread_id`. The second request uses a pronoun, so the Query Rewriter must resolve the referenced model from the first turn before document retrieval. Each turn performs query embedding, hybrid retrieval, reranking, and LLM generation against the existing index. The current `InMemorySaver` is not durable across Python processes.

In [5]:
thread_id = 'notebook-demo'
first_question = 'What classification accuracy did the ResNet26-V2 model achieve?'
second_question = 'What optimizer did it use?'

first_response = invoke_agent_graph(
    graph,
    first_question,
    thread_id=thread_id,
)

second_response = invoke_agent_graph(
    graph,
    second_question,
    thread_id=thread_id,
)

display(Markdown(f'**Turn 1 answer**: {first_response.answer}'))
display(Markdown(f'**Turn 2 answer**: {second_response.answer}'))
for source in second_response.sources:
    print(f'- {source.chunk_id} | {source.source} | page {source.page}')

2026-08-24 14:33:53,710 | INFO | src.generation.llm | Generated answer with 226 character(s)
2026-08-24 14:33:55,526 | INFO | src.generation.llm | Generated answer with 198 character(s)
2026-08-24 14:33:56,052 | INFO | src.retrieval.hybrid_retriever | Fused 40 dense and 40 BM25 result(s) into 30 chunk(s)
2026-08-24 14:33:56,155 | INFO | src.retrieval.reranker | Reranked 30 candidate(s) through ScaDS.AI
2026-08-24 14:34:07,199 | INFO | src.generation.llm | Generated answer with 1286 character(s)
2026-08-24 14:34:08,322 | INFO | src.generation.llm | Generated answer with 83 character(s)
2026-08-24 14:34:08,323 | INFO | src.generation.rag_pipeline | Answered question with 4 retrieved source(s)
2026-08-24 14:34:13,444 | INFO | src.generation.llm | Generated answer with 502 character(s)
2026-08-24 14:34:15,599 | INFO | src.generation.llm | Generated answer with 162 character(s)
2026-08-24 14:34:18,254 | INFO | src.generation.llm | Generated answer with 288 character(s)
2026-08-24 14:34:20,2

**Turn 1 answer**: The ResNet26-V2 model has a classification accuracy of 0.9434. HuMengqing_chunk_125

**Turn 2 answer**: The ResNet-V2 network used the Stochastic Gradient Descent (SGD) optimizer with momentum. HuMengqing_chunk_093

- HuMengqing_chunk_093 | HuMengqing.pdf | page 50


## Inspect checkpointed state

The checkpoint stores the state after graph execution, including the retrieval action, original and rewritten query, retrieved chunks, and bounded conversation history. Confirm that `original_query` is the second request and `rewritten_query` resolves `it` to `ResNet26-V2`.

In [6]:
graph_config = {'configurable': {'thread_id': thread_id}}
snapshot = graph.get_state(graph_config)
snapshot.values

{'question': 'What optimizer did it use?',
 'retrieval_action': 'retrieve',
 'retrieval_confidence': 0.8,
 'retrieval_reason': 'The request is asking for a specific detail about the ResNet26-V2 model, which is related to the previous question about its classification accuracy, and this information could plausibly be found in documents about the model.',
 'conversation_history': [{'role': 'user',
   'content': 'What classification accuracy did the ResNet26-V2 model achieve?'},
  {'role': 'assistant',
   'content': 'The ResNet26-V2 model has a classification accuracy of 0.9434. HuMengqing_chunk_125'},
  {'role': 'user', 'content': 'What optimizer did it use?'},
  {'role': 'assistant',
   'content': 'The ResNet-V2 network used the Stochastic Gradient Descent (SGD) optimizer with momentum. HuMengqing_chunk_093'}],
 'conversation_context': [{'role': 'user',
   'content': 'What classification accuracy did the ResNet26-V2 model achieve?'},
  {'role': 'assistant',
   'content': 'The ResNet26-V

## Check LangSmith configuration safely

This cell checks whether tracing is configured without printing the API key. When enabled, open the `doc-qa-agent` project in the LangSmith website to inspect the retrieval gate decision and selected graph branch.

In [7]:
import os


langsmith_status = {
    'tracing_enabled': os.getenv('LANGSMITH_TRACING', '').lower() == 'true',
    'api_key_configured': bool(os.getenv('LANGSMITH_API_KEY')),
    'project': os.getenv('LANGSMITH_PROJECT', 'default'),
}
langsmith_status

{'tracing_enabled': True,
 'api_key_configured': True,
 'project': 'doc-qa-agent'}

## Runtime requirements

Build the index first with `python -m scripts.build_index` whenever source documents, parsing, chunking, filtering, or the embedding model changes. The online notebook requires `SCADS_API_KEY` for reranking and generation. It needs `MINERU_API_TOKEN` only when building the index. Configure `LANGSMITH_TRACING=true`, `LANGSMITH_API_KEY`, and `LANGSMITH_PROJECT=doc-qa-agent` to view traces in LangSmith. Do not run indexing against sensitive documents unless uploading their content to MinerU and ScaDS.AI is permitted.